# Chapter 14 — RAG Fundamentals: The Theory Behind Semantic Search

**Volume 2: Practical Applications — AI for Networking and Security Engineers**

In this notebook you will build a complete Retrieval-Augmented Generation (RAG) pipeline from scratch.
You will see **why keyword search fails** network engineers, understand **vector embeddings** and **cosine similarity**,
learn the **chunking strategies** that make or break real systems, and finish with a working RAG Q&A bot
that answers questions from network documentation using Claude.

---
**What you will build — 5 hands-on demos:**

| # | Demo | Key concept |
|---|------|-------------|
| 1 | Keyword search vs. semantic search | Why grep is not enough |
| 2 | Chunking strategies | The most underestimated parameter in RAG |
| 3 | TF-IDF vector store (no external APIs) | Embeddings + cosine similarity from scratch |
| 4 | HyDE retrieval — query expansion | Finding docs that answer, not just match |
| 5 | Full RAG pipeline with Claude | Chunk → embed → retrieve → generate |

> **Prerequisite:** `!pip install anthropic scikit-learn numpy` (run the install cell first)


In [ ]:
!pip install -q anthropic scikit-learn numpy

---
## Demo 1 — Why Keyword Search Fails Network Engineers

Imagine it is 2 AM and a BGP session is down.
You search your Confluence for "BGP session down".
But the runbook says **"peer not establishing"** — no match.

Keyword search (like `grep`) requires **exact word overlap**.
Semantic search finds documents with the **same meaning**, even with different words.

This demo shows the gap between the two approaches on real networking phrases.


In [ ]:
# Demo 1: Keyword search vs semantic search
# We simulate a documentation database with 8 networking snippets.

DOCS = [
    # BGP-related
    "BGP peer not establishing — check hello timers and AS numbers",
    "Neighbor session is down due to TCP port 179 being blocked by ACL",
    "BGP route not received — verify advertise-map and prefix-list configuration",
    # OSPF-related
    "OSPF adjacency stuck in EXSTART — likely MTU mismatch between neighbors",
    "OSPF neighbor stays in INIT state — check hello and dead interval mismatch",
    "OSPF area 0 backbone must be contiguous for proper LSA flooding",
    # Unrelated
    "Interface GigabitEthernet0/1 is administratively down",
    "NTP server unreachable — check routing table and firewall rules",
]

# ── Keyword search (grep style) ─────────────────────────────────────────────
def keyword_search(query: str, docs: list, top_n: int = 3):
    """Simple keyword match — every word must appear in the doc."""
    query_words = set(query.lower().split())
    results = []
    for i, doc in enumerate(docs):
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        if overlap > 0:
            results.append((overlap, i, doc))
    results.sort(reverse=True)
    return results[:top_n]

# ── Semantic search with TF-IDF ─────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(ngram_range=(1, 2))   # unigrams + bigrams
doc_matrix = vectorizer.fit_transform(DOCS)

def semantic_search(query: str, top_n: int = 3):
    """TF-IDF + cosine similarity — finds semantically close docs."""
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, doc_matrix).flatten()
    top_idx = np.argsort(-scores)[:top_n]
    return [(scores[i], i, DOCS[i]) for i in top_idx if scores[i] > 0]

# ── Compare on three queries ─────────────────────────────────────────────────
queries = [
    "BGP session is down",            # paraphrase: "peer not establishing"
    "neighbors not talking to each other",   # abstract description
    "OSPF link state flood issue",    # "LSA flooding" is in the docs
]

for q in queries:
    print(f"\n{'='*65}")
    print(f'  Query: "{q}"')
    print(f"{'='*65}")

    kw = keyword_search(q, DOCS)
    print(f"\n  🔍 Keyword search ({len(kw)} hits):")
    if kw:
        for score, idx, doc in kw:
            print(f"     [{score} words] {doc[:80]}")
    else:
        print("     (no results)")

    sem = semantic_search(q)
    print(f"\n  🧠 Semantic search (top {len(sem)}):")
    for score, idx, doc in sem:
        print(f"     [{score:.3f}] {doc[:80]}")


---
## Demo 2 — Chunking Strategies

Chunking is **the most underestimated parameter in RAG**.
The right chunk size has more impact than the choice of embedding model.

**The core tradeoff:**
- Small chunks → precise embeddings, but may miss context
- Large chunks → rich context, but blurry embeddings
- Overlapping chunks → prevents answer "falling through the cracks"

Think of it like **BGP community tagging**:
- One community for the entire AS = too coarse (like embedding a whole document)
- Granular per-prefix tags = precise retrieval (like small focused chunks)

This demo shows three chunking strategies on the same networking runbook text.


In [ ]:
# Demo 2: Three chunking strategies compared

RUNBOOK = """
BGP Troubleshooting Runbook v2.3 — AS 65001

Section 1: Neighbor Not Establishing
When a BGP neighbor fails to establish, begin by verifying TCP connectivity.
BGP uses TCP port 179. Run: telnet <neighbor-ip> 179.
If the TCP connection fails, check interface status, routing table, and ACLs.
Next check AS numbers. A mismatch causes BGP OPEN messages to be rejected.
Use: show bgp neighbors | include remote AS.
Check the BGP timer configuration. Default hold time is 90 seconds, hello 30.
If either side uses a non-default value, both sides must agree.

Section 2: Routes Not Being Received
After the session establishes, verify route advertisements.
Use: show bgp ipv4 unicast neighbors <ip> received-routes.
If routes are missing, check prefix-list and route-map configurations.
Common issue: inbound route-map filtering out the expected prefixes.
Also check: maximum-prefix limits may be causing routes to be suppressed.

Section 3: Route Flapping
BGP route instability often has physical causes.
Check: show interface for input errors and CRC errors.
Duplex mismatches on the BGP link are a common cause of CRC errors.
Use: show bgp ipv4 unicast flap-statistics to identify unstable prefixes.
Consider implementing BGP dampening for unstable external prefixes.
"""

# ── Strategy 1: Fixed-size chunks (naive) ───────────────────────────────────
def chunk_fixed(text: str, chunk_size: int = 200) -> list:
    """Split at every N characters — simple but ignores sentence boundaries."""
    words = text.split()
    chunks = []
    current = []
    current_len = 0
    for w in words:
        current.append(w)
        current_len += len(w) + 1
        if current_len >= chunk_size:
            chunks.append(" ".join(current))
            current = []
            current_len = 0
    if current:
        chunks.append(" ".join(current))
    return chunks

# ── Strategy 2: Sentence-aware chunks with overlap ───────────────────────────
def chunk_with_overlap(text: str, chunk_size: int = 3, overlap: int = 1) -> list:
    """Split into sentence groups with overlap — prevents boundary splits."""
    sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 10]
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(sentences), step):
        group = sentences[i : i + chunk_size]
        chunks.append(". ".join(group) + ".")
    return chunks

# ── Strategy 3: Section-aware (hierarchical) ────────────────────────────────
def chunk_hierarchical(text: str) -> list:
    """Respect document structure — split at section headings."""
    chunks = []
    current_section = []
    for line in text.strip().splitlines():
        if line.startswith("Section"):
            if current_section:
                chunks.append("\n".join(current_section).strip())
            current_section = [line]
        else:
            current_section.append(line)
    if current_section:
        chunks.append("\n".join(current_section).strip())
    return [c for c in chunks if len(c) > 20]

# ── Compare all three ────────────────────────────────────────────────────────
print("=" * 65)
print("  CHUNKING STRATEGY COMPARISON")
print("=" * 65)

fixed = chunk_fixed(RUNBOOK, chunk_size=200)
overlap = chunk_with_overlap(RUNBOOK, chunk_size=3, overlap=1)
hier = chunk_hierarchical(RUNBOOK)

strategies = [
    ("Fixed-size (200 chars)", fixed),
    ("Sentence + overlap (3 sent, 1 overlap)", overlap),
    ("Hierarchical (section-aware)", hier),
]

for name, chunks in strategies:
    print(f"\n  Strategy: {name}")
    print(f"  Chunks produced: {len(chunks)}")
    print(f"  Avg chunk length: {sum(len(c) for c in chunks)//len(chunks)} chars")
    print(f"\n  First chunk preview:")
    print(f"  {repr(chunks[0][:120])}...")
    print()


---
## Demo 3 — Build a Mini Vector Store (No APIs, No Cloud)

Before we use neural embeddings, let's understand the **fundamentals**:
- What is a vector? (a list of numbers representing meaning)
- What is cosine similarity? (how close are two meaning-vectors)
- How does a vector store work? (a searchable database of meaning-vectors)

We build this from scratch with TF-IDF and NumPy.
**No API keys, no cloud, no external databases** — pure Python.

> **Networking analogy:** TF-IDF is like OSPF cost calculation.
> Rare words (high IDF) = low-cost preferred paths.
> Common words like "the" = high-cost paths, used as last resort.


In [ ]:
# Demo 3: Mini vector store built from scratch with TF-IDF + NumPy

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Network documentation chunks (our "knowledge base") ─────────────────────
KNOWLEDGE_BASE = [
    # BGP
    "BGP uses TCP port 179 to establish neighbor sessions between routers.",
    "BGP peer not establishing: verify AS numbers match with 'show bgp neighbors'.",
    "BGP hold timer is 90 seconds by default; hello packets sent every 30 seconds.",
    "Route reflectors eliminate the need for full-mesh iBGP in large networks.",
    "BGP communities tag routes with policies for filtering and traffic engineering.",
    # OSPF
    "OSPF hello interval defaults to 10 seconds on broadcast networks.",
    "OSPF area 0 is the backbone; all other areas must connect to area 0.",
    "OSPF MTU mismatch causes adjacency to get stuck in EXSTART or EXCHANGE state.",
    "OSPF uses SPF algorithm to compute shortest path tree from the LSDB.",
    "Stub areas reduce LSDB size by blocking external LSAs (Type 5).",
    # General
    "Interface duplex mismatch causes late collisions and CRC errors.",
    "NTP synchronization is critical for log correlation across network devices.",
    "ACLs are stateless; they do not track TCP session state like a firewall does.",
]

# ── Step 1: Build TF-IDF embeddings ─────────────────────────────────────────
print("Step 1: Building TF-IDF embeddings...")
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   # capture "BGP neighbor", "hello timer" as units
    sublinear_tf=True,    # dampen high-frequency terms
)
doc_vectors = vectorizer.fit_transform(KNOWLEDGE_BASE)
print(f"  Vocabulary size: {len(vectorizer.vocabulary_)} unique terms")
print(f"  Document matrix: {doc_vectors.shape[0]} docs × {doc_vectors.shape[1]} features")

# ── Step 2: Implement cosine similarity from scratch ─────────────────────────
def cosine_sim_manual(vec_a, vec_b):
    """Pure numpy cosine similarity — no sklearn."""
    a = vec_a.flatten()
    b = vec_b.flatten()
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

# ── Step 3: Search the vector store ─────────────────────────────────────────
def vector_store_search(query: str, top_k: int = 3) -> list:
    """Embed the query and find the most similar docs."""
    q_vec = vectorizer.transform([query]).toarray()
    db_array = doc_vectors.toarray()

    scores = []
    for i, doc_vec in enumerate(db_array):
        sim = cosine_sim_manual(q_vec, doc_vec)
        scores.append((sim, i, KNOWLEDGE_BASE[i]))

    scores.sort(reverse=True)
    return scores[:top_k]

# ── Step 4: Run searches ─────────────────────────────────────────────────────
print("\nStep 2: Searching the vector store...\n")
test_queries = [
    "BGP session keeps dropping",
    "OSPF neighbors not forming adjacency",
    "network interface errors and packet loss",
]

for q in test_queries:
    print(f'  Query: "{q}"')
    results = vector_store_search(q, top_k=3)
    for score, idx, doc in results:
        bar = "█" * int(score * 20)
        print(f"    [{score:.3f}] {bar:<20} {doc[:65]}...")
    print()

# ── Step 5: Show similarity matrix (concept visualization) ───────────────────
print("Step 3: Similarity between queries (concept space):\n")
sample_docs = [
    "BGP peer not establishing",
    "OSPF neighbor stuck in EXSTART",
    "interface CRC errors duplex mismatch",
]
sample_matrix = vectorizer.transform(sample_docs).toarray()
print(f"  {'':30} {'Doc1':>6} {'Doc2':>6} {'Doc3':>6}")
for i, d1 in enumerate(sample_docs):
    row = []
    for j, d2 in enumerate(sample_docs):
        sim = cosine_sim_manual(sample_matrix[i], sample_matrix[j])
        row.append(f"{sim:.3f}")
    print(f"  {d1[:30]:30} {row[0]:>6} {row[1]:>6} {row[2]:>6}")
print("\n  Scores near 1.0 = same topic | Scores near 0.0 = unrelated")


---
## Demo 4 — HyDE: Search With the Answer Shape, Not the Question

**The problem:** A short question and a long technical document look very different.
The query "BGP down?" and the runbook paragraph "BGP peer fails to establish due to TCP port 179 being blocked..." — these may be far apart in vector space even though one answers the other.

**HyDE (Hypothetical Document Embeddings)** solves this:
1. Ask the LLM to generate a **hypothetical answer** (even without knowing the real answer)
2. Embed the hypothetical answer instead of the raw question
3. The hypothetical answer "looks like" the documentation — closer in vector space

> **Networking analogy:** It's like policy-based routing.
> Instead of routing by source address (the question), you route by destination profile (the expected answer shape).

We use Claude Haiku as the hypothetical answer generator (it's fast and cheap).


In [ ]:
# Demo 4: HyDE — Hypothetical Document Embedding for better retrieval
import os
from anthropic import Anthropic
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Reuse the same knowledge base from Demo 3
KNOWLEDGE_BASE = [
    "BGP uses TCP port 179 to establish neighbor sessions between routers.",
    "BGP peer not establishing: verify AS numbers match with 'show bgp neighbors'.",
    "BGP hold timer is 90 seconds by default; hello packets sent every 30 seconds.",
    "Route reflectors eliminate the need for full-mesh iBGP in large networks.",
    "BGP communities tag routes with policies for filtering and traffic engineering.",
    "OSPF hello interval defaults to 10 seconds on broadcast networks.",
    "OSPF area 0 is the backbone; all other areas must connect to area 0.",
    "OSPF MTU mismatch causes adjacency to get stuck in EXSTART or EXCHANGE state.",
    "OSPF uses SPF algorithm to compute shortest path tree from the LSDB.",
    "Stub areas reduce LSDB size by blocking external LSAs (Type 5).",
    "Interface duplex mismatch causes late collisions and CRC errors.",
    "NTP synchronization is critical for log correlation across network devices.",
    "ACLs are stateless; they do not track TCP session state like a firewall does.",
]

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
client = Anthropic()
HAIKU = "claude-haiku-4-5-20251001"

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
doc_vectors = vectorizer.fit_transform(KNOWLEDGE_BASE)

def direct_search(query: str, top_k: int = 3) -> list:
    """Standard semantic search — embed the question directly."""
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, doc_vectors).flatten()
    top_idx = np.argsort(-scores)[:top_k]
    return [(scores[i], KNOWLEDGE_BASE[i]) for i in top_idx]

def hyde_search(query: str, top_k: int = 3) -> tuple:
    """HyDE — generate a hypothetical answer, then search with THAT."""
    # Step 1: generate a hypothetical answer
    prompt = (
        f"You are a network engineer. Write a 2-3 sentence technical answer "
        f"to this question, as if from a network operations runbook:\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    r = client.messages.create(
        model=HAIKU,
        max_tokens=120,
        messages=[{"role": "user", "content": prompt}],
    )
    hypothetical = resp.content[0].text.strip()

    # Step 2: embed the hypothetical answer and search
    h_vec = vectorizer.transform([hypothetical])
    scores = cosine_similarity(h_vec, doc_vectors).flatten()
    top_idx = np.argsort(-scores)[:top_k]
    results = [(scores[i], KNOWLEDGE_BASE[i]) for i in top_idx]
    return hypothetical, results

# ── Compare direct vs HyDE on ambiguous queries ──────────────────────────────
queries = [
    "My BGP is broken",                          # very vague
    "why won't my neighbors talk to each other", # no networking jargon
]

for q in queries:
    print(f"\n{'='*65}")
    print(f'  Query: "{q}"')
    print(f"{'='*65}")

    # Direct search
    print("\n  Direct search results:")
    for score, doc in direct_search(q):
        print(f"    [{score:.3f}] {doc[:70]}")

    # HyDE search
    hypothetical, hyde_results = hyde_search(q)
    print(f"\n  HyDE hypothetical answer (used for search):")
    print(f'    "{hypothetical[:120]}"')
    print(f"\n  HyDE search results:")
    for score, doc in hyde_results:
        print(f"    [{score:.3f}] {doc[:70]}")


---
## Demo 5 — Full RAG Pipeline: Chunk → Embed → Retrieve → Generate

This is the complete RAG system end-to-end:

```
Network Runbook (text)
        │
        ▼
   [Chunking]          ← split into manageable pieces
        │
        ▼
   [Embedding]         ← convert each chunk to a vector
        │
        ▼
   [Vector Store]      ← save all chunk vectors in memory
        │
  (query arrives)
        │
        ▼
   [Retrieval]         ← find top-K most similar chunks
        │
        ▼
   [Augmented Prompt]  ← combine query + retrieved chunks
        │
        ▼
   [Claude Sonnet]     ← generate a grounded answer
        │
        ▼
   [Answer + Sources]  ← cited, traceable, not hallucinated
```

Claude uses **only the retrieved context** — not its general training knowledge.
This prevents hallucination and keeps answers grounded in YOUR documentation.


In [ ]:
from anthropic import Anthropic
# Demo 5: Full RAG pipeline — chunk, embed, retrieve, generate with Claude
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SONNET = "claude-sonnet-4-20250514"
HAIKU  = "claude-haiku-4-5-20251001"
client = Anthropic()

# ── Step 1: Network documentation (our knowledge base) ──────────────────────
NETWORK_DOCS = [
    # BGP runbook
    """BGP Neighbor Troubleshooting.
    When a BGP neighbor is not establishing, first verify TCP connectivity to port 179.
    Use: telnet <neighbor-ip> 179. If it fails, check ACLs blocking port 179.
    Verify AS numbers match: show bgp neighbors | include remote AS.
    Check local and remote AS values in the BGP configuration.""",

    """BGP Timer Configuration.
    Default BGP hold timer is 90 seconds, keepalive is 30 seconds.
    Mismatched timers between peers cause session drops.
    The lower value is negotiated during OPEN message exchange.
    To check: show bgp neighbors | include hold time.""",

    """BGP Route Advertisement.
    Routes must be in the routing table before BGP can advertise them.
    Use network statement or redistribution to inject routes into BGP.
    Verify with: show bgp ipv4 unicast | include Network.
    Outbound route-maps can filter what is advertised to specific neighbors.""",

    # OSPF runbook
    """OSPF Neighbor States.
    A healthy OSPF adjacency reaches FULL state.
    If stuck in EXSTART: check MTU mismatch (ip mtu and ip ospf mtu-ignore).
    If stuck in INIT: the neighbor receives hellos but its own hellos are not received.
    Check hello and dead intervals: they must match on both sides.""",

    """OSPF Area Design.
    Area 0 is the backbone and must remain contiguous.
    All other areas must connect to area 0 directly or via virtual links.
    Stub areas block external LSAs (Type 5) reducing LSDB size.
    NSSA allows limited external route injection into stub-like areas.""",

    # General
    """Interface Troubleshooting.
    CRC errors indicate a physical layer problem: bad cable, SFP, or duplex mismatch.
    Check: show interface for input errors, CRC, and giants/runts.
    Duplex mismatch: one side full-duplex, other half-duplex — causes late collisions.
    Fix: set both sides to the same duplex with: duplex full.""",

    """NTP Configuration.
    All network devices must sync to the same NTP source for log correlation.
    Without NTP sync, syslog timestamps differ making troubleshooting harder.
    Configure: ntp server <ip> prefer. Verify: show ntp status.
    Stratum 1 = GPS clock, Stratum 2 = server synced to stratum 1.""",
]

# ── Step 2: Chunk the documents ──────────────────────────────────────────────
def chunk_document(doc: str, max_sentences: int = 3) -> list:
    """Split a document into overlapping sentence chunks."""
    sentences = [s.strip() for s in doc.replace("\n", " ").split(".") if len(s.strip()) > 15]
    chunks = []
    for i in range(0, len(sentences), max_sentences - 1):
        chunk = ". ".join(sentences[i : i + max_sentences]).strip()
        if chunk:
            chunks.append(chunk + ".")
    return chunks

all_chunks = []
for doc in NETWORK_DOCS:
    all_chunks.extend(chunk_document(doc))

print(f"Step 1: Chunked {len(NETWORK_DOCS)} documents into {len(all_chunks)} chunks")

# ── Step 3: Build vector store ───────────────────────────────────────────────
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_df=0.9)
chunk_vectors = vectorizer.fit_transform(all_chunks)
print(f"Step 2: Vector store built — {chunk_vectors.shape[0]} vectors, {chunk_vectors.shape[1]} features")

# ── Step 4: Retrieval function ───────────────────────────────────────────────
def retrieve(query: str, top_k: int = 3) -> list:
    """Find the top-K most relevant chunks for a query."""
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, chunk_vectors).flatten()
    top_idx = np.argsort(-scores)[:top_k]
    return [(scores[i], all_chunks[i]) for i in top_idx if scores[i] > 0.01]

# ── Step 5: RAG generation ───────────────────────────────────────────────────
def rag_answer(question: str, top_k: int = 3) -> dict:
    """Full RAG pipeline: retrieve context, generate grounded answer."""
    # Retrieve
    retrieved = retrieve(question, top_k=top_k)

    if not retrieved:
        return {"question": question, "answer": "No relevant documentation found.", "sources": []}

    # Build augmented prompt
    context_block = "\n\n".join(
        f"[Source {i+1}] {chunk}" for i, (score, chunk) in enumerate(retrieved)
    )
    system_prompt = (
        "You are a network operations assistant. Answer the engineer's question "
        "using ONLY the provided documentation context. "
        "If the context does not contain the answer, say so. "
        "Be concise and practical. Cite which source you used."
    )
    user_prompt = f"Documentation context:\n{context_block}\n\nQuestion: {question}"

    # Generate
    resp = client.messages.create(
        model=SONNET,
        max_tokens=300,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )

    return {
        "question": question,
        "answer": resp.content[0].text.strip(),
        "sources": [chunk[:80] + "..." for _, chunk in retrieved],
        "retrieval_scores": [round(s, 3) for s, _ in retrieved],
    }

# ── Step 6: Ask the RAG system ───────────────────────────────────────────────
print("\nStep 3: Running RAG Q&A...\n")

questions = [
    "My BGP neighbor is not establishing — what do I check first?",
    "OSPF adjacency is stuck in EXSTART state — what is the cause?",
    "Why do I see CRC errors on my interfaces?",
]

for q in questions:
    print(f"\n{'='*65}")
    print(f"  Q: {q}")
    print(f"{'='*65}")
    result = rag_answer(q)
    print(f"\n  Answer:\n  {result['answer']}\n")
    print(f"  Retrieved sources (scores: {result['retrieval_scores']}):")
    for src in result["sources"]:
        print(f"    • {src}")


---
## Bonus: RAG Failure Mode Diagnostic

When your RAG system gives wrong answers, it's almost always one of these four failures:

| Failure | Symptom | Network analogy | Fix |
|---------|---------|----------------|-----|
| **Retrieval miss** | "I couldn't find info about this" | Route not in routing table | Better chunking, more overlap |
| **Contextual hallucination** | Plausible but wrong for YOUR network | Router ignoring routing table, making up paths | Stronger grounding prompt |
| **Chunking split failure** | Partial answer, missing procedure steps | PMTUD failure, packet doesn't reassemble | Larger overlap between chunks |
| **Redundancy noise** | Contradictory / confused answers | Routing loop — same prefix advertised twice | Deduplicate retrieved chunks |

The code below simulates and diagnoses each failure mode.


In [ ]:
# Bonus: Diagnose RAG failure modes

# ── Failure Mode 1: Retrieval Miss ──────────────────────────────────────────
# Happens when the query uses different terminology than the docs

TINY_KB = [
    "The BGP peer session has transitioned to the Established state.",
    "OSPF adjacency reached FULL state successfully.",
]

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vec_small = TfidfVectorizer()
mat_small = vec_small.fit_transform(TINY_KB)

def diagnose_retrieval_miss(query: str, threshold: float = 0.05):
    """Check if any doc crosses the relevance threshold."""
    q_vec = vec_small.transform([query])
    scores = cosine_similarity(q_vec, mat_small).flatten()
    best_score = max(scores)
    best_doc = TINY_KB[np.argmax(scores)]
    if best_score < threshold:
        print(f"  ❌ RETRIEVAL MISS: best score {best_score:.3f} < {threshold}")
        print(f"     Query used terminology not in the knowledge base.")
        print(f"     Fix: add synonyms, use multi-query retrieval, or fine-tune embeddings.")
    else:
        print(f"  ✅ Retrieved: [{best_score:.3f}] {best_doc[:60]}...")

print("=== Failure Mode 1: Retrieval Miss ===")
diagnose_retrieval_miss("neighbor won't talk to me")   # jargon mismatch
diagnose_retrieval_miss("BGP peer established")          # matches

# ── Failure Mode 2: Hallucination Guard ─────────────────────────────────────
# Check if the answer is "grounded" — i.e., key phrases appear in retrieved docs

def check_grounding(answer: str, retrieved_docs: list) -> bool:
    """Rough check: do key answer phrases appear in any retrieved doc?"""
    all_context = " ".join(retrieved_docs).lower()
    answer_words = set(answer.lower().split())
    stop = {"the", "a", "an", "is", "in", "of", "to", "and", "or", "it", "be"}
    key_words = [w for w in answer_words if len(w) > 4 and w not in stop]
    matched = sum(1 for w in key_words if w in all_context)
    ratio = matched / len(key_words) if key_words else 0
    return ratio

print("\n=== Failure Mode 2: Hallucination Check ===")
retrieved = ["BGP peer not establishing: verify AS numbers match."]
grounded_answer = "Check that AS numbers match using show bgp neighbors."
hallucinated_answer = "This is caused by cosmic rays affecting the router silicon."

for ans in [grounded_answer, hallucinated_answer]:
    ratio = check_grounding(ans, retrieved)
    icon = "✅" if ratio > 0.3 else "❌"
    print(f'  {icon} Grounding score: {ratio:.2f} — "{ans[:60]}"')

# ── Failure Mode 4: Redundancy Detection ────────────────────────────────────
print("\n=== Failure Mode 4: Redundancy Noise ===")
chunks = [
    "BGP uses TCP port 179 for neighbor sessions.",
    "BGP establishes sessions over TCP using port 179.",   # near-duplicate
    "OSPF uses multicast 224.0.0.5 for hello packets.",
]

vec_red = TfidfVectorizer(ngram_range=(1, 2))
mat_red = vec_red.fit_transform(chunks).toarray()

print("  Pairwise similarity matrix:")
print(f"  {'':>6} {'Chunk0':>8} {'Chunk1':>8} {'Chunk2':>8}")
for i in range(3):
    row = []
    for j in range(3):
        sim = np.dot(mat_red[i], mat_red[j]) / (
            np.linalg.norm(mat_red[i]) * np.linalg.norm(mat_red[j]) + 1e-9
        )
        row.append(f"{sim:.3f}")
    print(f"  Chunk{i}: {row[0]:>8} {row[1]:>8} {row[2]:>8}")

print("\n  Chunks 0 and 1 are near-duplicates (score near 1.0).")
print("  Fix: deduplicate before sending to LLM — keep only the highest-scoring unique chunk.")

print("\n✅ All RAG failure modes demonstrated. Use these patterns to diagnose your own systems.")


---
## Summary — What You Built

In this notebook you went from keyword search all the way to a production-ready RAG pipeline:

### Demo progression:
1. **Keyword vs. Semantic Search** — proved that `grep` misses semantically equivalent phrases
2. **Chunking Strategies** — compared fixed-size, overlapping, and hierarchical chunking
3. **TF-IDF Vector Store** — built a searchable vector database from pure Python
4. **HyDE Retrieval** — used Claude Haiku to generate a "hypothetical answer" for better matching
5. **Full RAG Pipeline** — complete end-to-end: chunk → embed → retrieve → Claude Sonnet generates

### Key formulas to remember:

**Cosine similarity:**
```
similarity(A, B) = (A · B) / (|A| × |B|)
```
> 0.8+ = direct match | 0.5–0.8 = related | <0.5 = probably irrelevant

**RAG quality equation:**
```
answer_quality = min(retrieval_quality, generation_quality)
```
> The weakest link wins. Fix retrieval before optimizing the LLM.

### In the next chapter:
Chapter 15 moves from TF-IDF to **neural embeddings** and **proper vector databases** (ChromaDB, FAISS).
The concepts you learned here — chunking, similarity search, grounding — apply directly to those advanced systems.

---
*Chapter 14 — RAG Fundamentals | AI for Networking and Security Engineers*
